# Silver - feriados

Desenvolvido por: Ygor Moraes

Este notebook cria a Silver auxiliar de feriados nacionais para enriquecer as Golds de entregas.

Regras aplicadas:
- gerar apenas feriados nacionais previstos em lei federal;
- remover pontos facultativos e feriados religiosos municipais;
- manter uma linha por `data_feriado`;
- gravar em Delta particionado por `ano` e `mes`;
- permitir join seguro com `dt_evento` da Silver de rastreamento.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Silver de feriados.

from datetime import date

from pyspark.sql.functions import (
    col,
    count,
    when,
    current_timestamp,
    to_date,
    lit,
    min as spark_min,
    max as spark_max
)

from pyspark.sql.types import (
    StructType,
    StructField,
    DateType,
    StringType,
    IntegerType
)

SILVER_RASTREAMENTO_TABLE = "ecommerce_rastreamento_entregas"
SILVER_FERIADOS_TABLE = "feriados"

SILVER_RASTREAMENTO_PATH = f"{SILVER_BASE_PATH}{SILVER_RASTREAMENTO_TABLE}"
SILVER_FERIADOS_PATH = f"{SILVER_BASE_PATH}{SILVER_FERIADOS_TABLE}"

SILVER_WRITE_MODE = "overwrite"

ANO_FINAL_FIXO = 2080

RASTREAMENTO_REQUIRED_COLUMNS = [
    "dt_evento",
    "ano",
    "mes"
]

FERIADOS_REQUIRED_COLUMNS = [
    "data_feriado",
    "nome_feriado",
    "tipo_feriado",
    "categoria_feriado",
    "ano",
    "mes",
    "silver_processed_at"
]

adls_options = get_adls_options()

print("Notebook configurado.")
print(f"Origem rastreamento: {SILVER_RASTREAMENTO_PATH}")
print(f"Destino feriados: {SILVER_FERIADOS_PATH}")
print(f"Modo de escrita: {SILVER_WRITE_MODE}")

In [0]:
# Lê a Silver de rastreamento e define o intervalo de anos para gerar feriados.

df_rastreamento = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_RASTREAMENTO_PATH)
)

rastreamento_columns = df_rastreamento.columns

missing_rastreamento_columns = [
    c for c in RASTREAMENTO_REQUIRED_COLUMNS
    if c not in rastreamento_columns
]

if missing_rastreamento_columns:
    raise Exception(f"Colunas obrigatórias ausentes no rastreamento: {missing_rastreamento_columns}")

df_anos_rastreamento = df_rastreamento.select(
    spark_min(col("ano")).alias("ano_min"),
    spark_max(col("ano")).alias("ano_max")
)

display(df_anos_rastreamento)

anos_rastreamento = df_anos_rastreamento.collect()[0]

ANO_INICIAL = int(anos_rastreamento["ano_min"])
ANO_FINAL = ANO_FINAL_FIXO

if ANO_FINAL < ANO_INICIAL:
    raise Exception("Erro: ANO_FINAL não pode ser menor que ANO_INICIAL.")

ANOS_FERIADOS = list(range(ANO_INICIAL, ANO_FINAL + 1))

total_rastreamento = df_rastreamento.count()

print("Silver de rastreamento lida com sucesso.")
print(f"Total de registros no rastreamento: {total_rastreamento}")
print(f"Anos dos feriados: {ANO_INICIAL} até {ANO_FINAL}")

In [0]:
# Gera apenas feriados nacionais previstos em lei federal.

def gerar_feriados_ano(ano):
    feriados = [
        (date(ano, 1, 1), "Confraternização Universal", "nacional", "fixo"),
        (date(ano, 4, 21), "Tiradentes", "nacional", "fixo"),
        (date(ano, 5, 1), "Dia Mundial do Trabalho", "nacional", "fixo"),
        (date(ano, 9, 7), "Independência do Brasil", "nacional", "fixo"),
        (date(ano, 10, 12), "Nossa Senhora Aparecida", "nacional", "fixo"),
        (date(ano, 11, 2), "Finados", "nacional", "fixo"),
        (date(ano, 11, 15), "Proclamação da República", "nacional", "fixo"),
        (date(ano, 12, 25), "Natal", "nacional", "fixo")
    ]

    if ano >= 2024:
        feriados.append(
            (date(ano, 11, 20), "Dia Nacional de Zumbi e da Consciência Negra", "nacional", "fixo")
        )

    return feriados

In [0]:
# Cria a Silver de feriados em memória com schema padronizado.

linhas_feriados = []

for ano in ANOS_FERIADOS:
    for data_feriado, nome_feriado, tipo_feriado, categoria_feriado in gerar_feriados_ano(ano):
        linhas_feriados.append((
            data_feriado,
            nome_feriado,
            tipo_feriado,
            categoria_feriado,
            ano,
            data_feriado.month
        ))

schema_feriados = StructType([
    StructField("data_feriado", DateType(), False),
    StructField("nome_feriado", StringType(), False),
    StructField("tipo_feriado", StringType(), False),
    StructField("categoria_feriado", StringType(), False),
    StructField("ano", IntegerType(), False),
    StructField("mes", IntegerType(), False)
])

df_feriados = (
    spark.createDataFrame(linhas_feriados, schema_feriados)
    .withColumn("silver_processed_at", current_timestamp())
)

total_feriados = df_feriados.count()

print("Silver de feriados criada em memória.")
print(f"Total de datas de feriado: {total_feriados}")

display(df_feriados.orderBy("data_feriado"))

In [0]:
# Valida schema, nulos e duplicidade da Silver de feriados em memória.

colunas_feriados = df_feriados.columns

colunas_ausentes = [
    c for c in FERIADOS_REQUIRED_COLUMNS
    if c not in colunas_feriados
]

if colunas_ausentes:
    raise Exception(f"Erro: colunas obrigatórias ausentes: {colunas_ausentes}")

duplicados_data_feriado = (
    df_feriados
    .groupBy("data_feriado")
    .count()
    .filter(col("count") > 1)
    .count()
)

df_validacao_feriados = df_feriados.select(
    count("*").alias("total_linhas"),
    count(when(col("data_feriado").isNull(), True)).alias("data_feriado_nulo"),
    count(when(col("nome_feriado").isNull(), True)).alias("nome_feriado_nulo"),
    count(when(col("tipo_feriado").isNull(), True)).alias("tipo_feriado_nulo"),
    count(when(col("categoria_feriado").isNull(), True)).alias("categoria_feriado_nulo"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),
    count(when(col("silver_processed_at").isNull(), True)).alias("silver_processed_at_nulo")
)

display(df_validacao_feriados)

print(f"Total de feriados em memória: {total_feriados}")
print(f"Datas de feriado duplicadas: {duplicados_data_feriado}")

if duplicados_data_feriado > 0:
    raise Exception("Erro: existem datas duplicadas na tabela de feriados.")

validacao_feriados = df_validacao_feriados.collect()[0]

if validacao_feriados["data_feriado_nulo"] > 0:
    raise Exception("Erro: existem registros com data_feriado nula.")

if validacao_feriados["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo.")

if validacao_feriados["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo.")

if validacao_feriados["silver_processed_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem silver_processed_at.")

print("Validação OK: Silver de feriados em memória aprovada.")

In [0]:
# Regrava a Silver de feriados em Delta, mantendo particionamento por ano e mes.

(
    df_feriados
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_FERIADOS_PATH)
)

print(f"Silver de feriados gravada com sucesso em: {SILVER_FERIADOS_PATH}")

In [0]:
# Lê a Silver gravada e valida volume, duplicidade e schema final.

df_feriados_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_FERIADOS_PATH)
)

total_feriados_saved = df_feriados_saved.count()

duplicados_data_feriado_saved = (
    df_feriados_saved
    .groupBy("data_feriado")
    .count()
    .filter(col("count") > 1)
    .count()
)

colunas_feriados_saved = df_feriados_saved.columns

colunas_ausentes_saved = [
    c for c in FERIADOS_REQUIRED_COLUMNS
    if c not in colunas_feriados_saved
]

print(f"Total feriados em memória: {total_feriados}")
print(f"Total feriados gravados: {total_feriados_saved}")
print(f"Datas duplicadas na Silver gravada: {duplicados_data_feriado_saved}")

if total_feriados_saved != total_feriados:
    raise Exception("Erro: quantidade gravada diferente da quantidade em memória.")

if duplicados_data_feriado_saved > 0:
    raise Exception("Erro: existem datas duplicadas na Silver de feriados gravada.")

if colunas_ausentes_saved:
    raise Exception(f"Erro: colunas obrigatórias ausentes na Silver gravada: {colunas_ausentes_saved}")

df_feriados_saved.printSchema()

print("Validação OK: Silver de feriados gravada corretamente.")

In [0]:
# Testa se o join com feriados não infla linhas no rastreamento.

df_rastreamento_datas = (
    df_rastreamento
    .withColumn("data_evento", to_date(col("dt_evento")))
)

df_feriados_join = (
    df_feriados_saved
    .select(
        "data_feriado",
        "nome_feriado",
        "tipo_feriado"
    )
    .distinct()
)

df_teste_join = (
    df_rastreamento_datas
    .join(
        df_feriados_join,
        df_rastreamento_datas["data_evento"] == df_feriados_join["data_feriado"],
        "left"
    )
    .withColumn(
        "evento_em_feriado",
        when(col("data_feriado").isNotNull(), lit(1)).otherwise(lit(0))
    )
)

total_rastreamento_antes_join = df_rastreamento_datas.count()
total_rastreamento_depois_join = df_teste_join.count()

print(f"Total rastreamento antes do join: {total_rastreamento_antes_join}")
print(f"Total rastreamento depois do join: {total_rastreamento_depois_join}")

if total_rastreamento_depois_join != total_rastreamento_antes_join:
    raise Exception("Erro: o join com feriados inflou a quantidade de linhas.")

display(
    df_teste_join
    .groupBy("evento_em_feriado")
    .count()
    .orderBy("evento_em_feriado")
)

print("Validação OK: join com feriados não inflou linhas.")